# Module 4 - Topic 2: Metadata Filtering & Re-ranking
**AI Application Development - Developer Track (NIGCOMSAT)**

In this demo you ingest the NovaNet HR corpus into ChromaDB with metadata, show why unfiltered similarity search returns wrong answers, add metadata filters and a rule-based re-ranker, and measure the improvement on a fixed test set. One concept per cell; run them in order.

## Install
Run this once if these packages are not already installed. Remove the leading `#` to run, then add it back.

In [ ]:
# !pip install chromadb openai python-dotenv

## Import & Setup
We load credentials from `.env` (see `.env.example` at the repository root), create an embedding function that calls the embedding API configured in your `.env` (`EMBEDDING_BASE_URL`, `EMBEDDING_API_KEY`, `EMBEDDING_MODEL`), and set up a persistent ChromaDB client so the collection survives notebook restarts.

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings

load_dotenv()

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

embedding_client = OpenAI(
    api_key=os.getenv("EMBEDDING_API_KEY"),
    base_url=os.getenv("EMBEDDING_BASE_URL"),
)


class CorpusEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        pass

    def __call__(self, input: Documents) -> Embeddings:
        response = embedding_client.embeddings.create(
            model=EMBEDDING_MODEL, input=input
        )
        return [item.embedding for item in response.data]


embedding_fn = CorpusEmbeddingFunction()

chroma_client = chromadb.PersistentClient(path="./chroma_db")

print("ChromaDB client ready (persistent storage: ./chroma_db)")
print(f"Embedding model: {EMBEDDING_MODEL}")

ChromaDB client ready (persistent storage: ./chroma_db)
Embedding model: Qwen/Qwen3-Embedding-0.6B


## Corpus Metadata Registry
Every document in the NovaNet HR corpus carries metadata that we will store alongside its vector in ChromaDB. We define the metadata here in a registry keyed by filename, rather than parsing it from the document headers, so the values are explicit and reviewable.

**Note:** `date_issued` is stored as an integer in `YYYYMMDD` format (e.g. `20250701`) because ChromaDB's comparison operators (`$gte`, `$lte`) require numeric values — they do not work on strings.

In [7]:
CORPUS_METADATA = {
    "HR-POL-2024-001_annual_leave_policy.txt": {
        "doc_id": "HR-POL-2024-001", "doc_type": "policy", "department": "HR",
        "date_issued": 20240115, "status": "superseded",
        "title": "Annual Leave Policy"
    },
    "HR-POL-2024-002_code_of_conduct.txt": {
        "doc_id": "HR-POL-2024-002", "doc_type": "policy", "department": "HR",
        "date_issued": 20240201, "status": "current",
        "title": "Code of Conduct"
    },
    "HR-POL-2024-003_remote_work_policy.txt": {
        "doc_id": "HR-POL-2024-003", "doc_type": "policy", "department": "HR / IT",
        "date_issued": 20240301, "status": "current",
        "title": "Remote Work Policy"
    },
    "HR-POL-2025-004_data_protection_policy.txt": {
        "doc_id": "HR-POL-2025-004", "doc_type": "policy", "department": "HR / Legal",
        "date_issued": 20250110, "status": "current",
        "title": "Data Protection Policy"
    },
    "HR-POL-2025-005_travel_expense_policy.txt": {
        "doc_id": "HR-POL-2025-005", "doc_type": "policy", "department": "HR / Finance",
        "date_issued": 20250301, "status": "current",
        "title": "Travel and Expense Policy"
    },
    "HR-POL-2025-006_annual_leave_policy_revised.txt": {
        "doc_id": "HR-POL-2025-006", "doc_type": "policy", "department": "HR",
        "date_issued": 20250701, "status": "current",
        "title": "Annual Leave Policy (Revised)"
    },
    "HR-SOP-2024-001_employee_onboarding.txt": {
        "doc_id": "HR-SOP-2024-001", "doc_type": "sop", "department": "HR",
        "date_issued": 20240401, "status": "current",
        "title": "Employee Onboarding Procedure"
    },
    "HR-SOP-2025-002_performance_review.txt": {
        "doc_id": "HR-SOP-2025-002", "doc_type": "sop", "department": "HR",
        "date_issued": 20250115, "status": "current",
        "title": "Performance Review Procedure"
    },
    "HR-SOP-2025-003_employee_exit.txt": {
        "doc_id": "HR-SOP-2025-003", "doc_type": "sop", "department": "HR",
        "date_issued": 20250401, "status": "current",
        "title": "Employee Exit Procedure"
    },
    "HR-SOP-2025-004_expense_claims.txt": {
        "doc_id": "HR-SOP-2025-004", "doc_type": "sop", "department": "HR / Finance",
        "date_issued": 20250501, "status": "current",
        "title": "Expense Claims Procedure"
    },
    "HR-CIR-2025-001_revised_leave_entitlements.txt": {
        "doc_id": "HR-CIR-2025-001", "doc_type": "circular", "department": "HR",
        "date_issued": 20250615, "status": "current",
        "title": "Revised Leave Entitlements"
    },
    "HR-CIR-2025-002_public_holiday_schedule.txt": {
        "doc_id": "HR-CIR-2025-002", "doc_type": "circular", "department": "HR",
        "date_issued": 20251128, "status": "current",
        "title": "2026 Public Holiday Schedule"
    },
    "HR-CIR-2026-003_remote_work_allowance_update.txt": {
        "doc_id": "HR-CIR-2026-003", "doc_type": "circular", "department": "HR / IT",
        "date_issued": 20260210, "status": "current",
        "title": "Updated Remote Work Equipment Allowance"
    },
    "HR-FAQ-2025-001_leave_and_absence.txt": {
        "doc_id": "HR-FAQ-2025-001", "doc_type": "faq", "department": "HR",
        "date_issued": 20250801, "status": "current",
        "title": "FAQ: Leave and Absence"
    },
    "HR-FAQ-2025-002_benefits_and_allowances.txt": {
        "doc_id": "HR-FAQ-2025-002", "doc_type": "faq", "department": "HR",
        "date_issued": 20250901, "status": "current",
        "title": "FAQ: Benefits and Allowances"
    },
}

print(f"{len(CORPUS_METADATA)} documents in registry")
for fname, m in CORPUS_METADATA.items():
    print(f"  {m['doc_id']:20s}  {m['status']:12s}  {m['doc_type']:10s}  {m['title']}")

15 documents in registry
  HR-POL-2024-001       superseded    policy      Annual Leave Policy
  HR-POL-2024-002       current       policy      Code of Conduct
  HR-POL-2024-003       current       policy      Remote Work Policy
  HR-POL-2025-004       current       policy      Data Protection Policy
  HR-POL-2025-005       current       policy      Travel and Expense Policy
  HR-POL-2025-006       current       policy      Annual Leave Policy (Revised)
  HR-SOP-2024-001       current       sop         Employee Onboarding Procedure
  HR-SOP-2025-002       current       sop         Performance Review Procedure
  HR-SOP-2025-003       current       sop         Employee Exit Procedure
  HR-SOP-2025-004       current       sop         Expense Claims Procedure
  HR-CIR-2025-001       current       circular    Revised Leave Entitlements
  HR-CIR-2025-002       current       circular    2026 Public Holiday Schedule
  HR-CIR-2026-003       current       circular    Updated Remote Work Equipme

## Load, Chunk, and Ingest
We read each `.txt` file, split it into sections using the `----` separator, discard the metadata header, and store the remaining sections as chunks in ChromaDB with the metadata from our registry. This is the ingestion step that the reference application's `ingest.py` will do once.

In [8]:
CORPUS_DIR = os.path.join("..", "..", "..", "data", "synthetic-hr-corpus")
SEPARATOR = "-" * 60

def load_and_chunk(corpus_dir):
    all_chunks = []
    for filename, meta in CORPUS_METADATA.items():
        filepath = os.path.join(corpus_dir, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        sections = text.split(SEPARATOR)
        for i, section in enumerate(sections[1:], start=1):
            section_text = section.strip()
            if len(section_text) < 30:
                continue
            all_chunks.append({
                "id": f"{meta['doc_id']}_chunk_{i}",
                "text": section_text,
                "metadata": {
                    "doc_id": meta["doc_id"],
                    "doc_type": meta["doc_type"],
                    "department": meta["department"],
                    "date_issued": meta["date_issued"],
                    "status": meta["status"],
                    "title": meta["title"],
                },
            })
    return all_chunks

chunks = load_and_chunk(CORPUS_DIR)
print(f"Loaded {len(chunks)} chunks from {len(CORPUS_METADATA)} documents\n")
for doc_id in sorted(set(c["metadata"]["doc_id"] for c in chunks)):
    n = sum(1 for c in chunks if c["metadata"]["doc_id"] == doc_id)
    print(f"  {doc_id}: {n} chunk{'s' if n > 1 else ''}")

Loaded 90 chunks from 15 documents

  HR-CIR-2025-001: 1 chunk
  HR-CIR-2025-002: 1 chunk
  HR-CIR-2026-003: 1 chunk
  HR-FAQ-2025-001: 2 chunks
  HR-FAQ-2025-002: 2 chunks
  HR-POL-2024-001: 7 chunks
  HR-POL-2024-002: 6 chunks
  HR-POL-2024-003: 11 chunks
  HR-POL-2025-004: 8 chunks
  HR-POL-2025-005: 10 chunks
  HR-POL-2025-006: 13 chunks
  HR-SOP-2024-001: 7 chunks
  HR-SOP-2025-002: 6 chunks
  HR-SOP-2025-003: 9 chunks
  HR-SOP-2025-004: 6 chunks


In [9]:
collection = chroma_client.get_or_create_collection(
    name="novanet_hr",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0:
    collection.add(
        ids=[c["id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        metadatas=[c["metadata"] for c in chunks],
    )
    print(f"Ingested {len(chunks)} chunks into collection 'novanet_hr'")
else:
    print(f"Collection already has {collection.count()} chunks -- skipping ingestion")

Collection already has 90 chunks -- skipping ingestion


## The Problem: Querying Without Filters
We query for annual leave entitlement with no metadata filters. The superseded policy (HR-POL-2024-001, 20 days) may score higher than the current one (HR-POL-2025-006, 25 days) because its wording is a closer semantic match.

In [10]:
def print_results(results, label="Results"):
    """Pretty-print ChromaDB query results."""
    print(f"\n{label}")
    print("=" * 90)
    for i, (chunk_id, doc, meta, dist) in enumerate(
        zip(results["ids"][0], results["documents"][0],
            results["metadatas"][0], results["distances"][0]), 1
    ):
        sim = 1 - dist
        print(f"  {i}. [{meta['doc_id']}] {meta['title']}")
        print(f"     Status: {meta['status']}  |  Type: {meta['doc_type']}  |  Similarity: {sim:.3f}")
        preview = doc.replace('\n', ' ')[:150]
        print(f"     {preview}...")
        print()


query = "How many days of annual leave do I get?"

results_raw = collection.query(query_texts=[query], n_results=5)
print_results(results_raw, f'Query: "{query}" -- NO FILTERS')


Query: "How many days of annual leave do I get?" -- NO FILTERS
  1. [HR-FAQ-2025-001] FAQ: Leave and Absence
     Status: current  |  Type: faq  |  Similarity: 0.640
     Q1: How many days of annual leave do I get? A1: Confirmed full-time employees get 25 working days per year.     This was increased from 20 days in Jul...

  2. [HR-POL-2024-001] Annual Leave Policy
     Status: superseded  |  Type: policy  |  Similarity: 0.593
     (a) All confirmed employees are entitled to 20 working days of     paid annual leave per calendar year.  (b) Employees who join part-way through the y...

  3. [HR-POL-2025-006] Annual Leave Policy (Revised)
     Status: current  |  Type: policy  |  Similarity: 0.583
     In addition to annual leave, employees are entitled to:  (a) 5 working days of compassionate leave on the death of an     immediate family member (spo...

  4. [HR-POL-2025-006] Annual Leave Policy (Revised)
     Status: current  |  Type: policy  |  Similarity: 0.578
     (a) All confirme

## Fix 1: Filtering by Status
One line changes the results from wrong to right. The `where` parameter filters by metadata *before* the similarity search, so superseded documents never enter the candidate set.

In [11]:
query = "How many days of annual leave do I get?"

results_filtered = collection.query(
    query_texts=[query],
    n_results=5,
    where={"status": "current"}
)
print_results(results_filtered, f'Query: "{query}" -- status=current')


Query: "How many days of annual leave do I get?" -- status=current
  1. [HR-FAQ-2025-001] FAQ: Leave and Absence
     Status: current  |  Type: faq  |  Similarity: 0.640
     Q1: How many days of annual leave do I get? A1: Confirmed full-time employees get 25 working days per year.     This was increased from 20 days in Jul...

  2. [HR-POL-2025-006] Annual Leave Policy (Revised)
     Status: current  |  Type: policy  |  Similarity: 0.583
     In addition to annual leave, employees are entitled to:  (a) 5 working days of compassionate leave on the death of an     immediate family member (spo...

  3. [HR-POL-2025-006] Annual Leave Policy (Revised)
     Status: current  |  Type: policy  |  Similarity: 0.578
     (a) All confirmed employees are entitled to 25 working days of     paid annual leave per calendar year.  (b) Employees in their first year of employme...

  4. [HR-CIR-2025-001] Revised Leave Entitlements
     Status: current  |  Type: circular  |  Similarity: 0.555
     Dear C

## Fix 2: Compound Filters
ChromaDB supports `$and` and `$or` to combine conditions. Here we filter for current documents issued from 2025 onwards, excluding FAQs, to answer a question about recent policy changes.

In [12]:
query = "What changed in the leave policy this year?"

results_compound = collection.query(
    query_texts=[query],
    n_results=5,
    where={
        "$and": [
            {"status": "current"},
            {"$and": [
                {"date_issued": {"$gte": 20250101}},
                {"doc_type": {"$ne": "faq"}}
            ]}
        ]
    }
)
print_results(results_compound, f'Query: "{query}" -- current + date>=2025 + not FAQ')


Query: "What changed in the leave policy this year?" -- current + date>=2025 + not FAQ
  1. [HR-POL-2025-006] Annual Leave Policy (Revised)
     Status: current  |  Type: policy  |  Similarity: 0.666
     10. WHAT CHANGED FROM THE PREVIOUS POLICY (HR-POL-2024-001)...

  2. [HR-POL-2025-006] Annual Leave Policy (Revised)
     Status: current  |  Type: policy  |  Similarity: 0.656
     This policy replaces HR-POL-2024-001 and sets out the updated entitlements and procedures for annual leave for all employees of NovaNet Technologies L...

  3. [HR-CIR-2025-001] Revised Leave Entitlements
     Status: current  |  Type: circular  |  Similarity: 0.538
     Dear Colleagues,  Following a review of our leave benefits, Management has approved the following changes, effective 1 July 2025:  1. ANNUAL LEAVE    ...

  4. [HR-POL-2025-006] Annual Leave Policy (Revised)
     Status: current  |  Type: policy  |  Similarity: 0.529
     - Annual leave increased from 20 to 25 working days - Carry-forward

## The Trap Filters Cannot Catch
The remote work policy (HR-POL-2024-003) says **NGN 25,000/month**. A later circular (HR-CIR-2026-003) updated the amount to **NGN 35,000/month** without replacing the policy. Both are status `current`. A status filter alone will not fix this -- the system must recognise that the circular is more recent.

In [17]:
query = "What is the monthly remote work allowance?"

results_trap = collection.query(
    query_texts=[query],
    n_results=5,
    where={"status": "current"}
)
print_results(results_trap, f'Query: "{query}" -- status=current (THE TRAP)')

print("CHECK: Does the circular (HR-CIR-2026-003, NGN 35,000) rank above")
print("       the policy (HR-POL-2024-003, NGN 25,000)?")
print("       If not, the re-ranker needs to fix the ordering.")


Query: "What is the monthly remote work allowance?" -- status=current (THE TRAP)
  1. [HR-CIR-2026-003] Updated Remote Work Equipment Allowance
     Status: current  |  Type: circular  |  Similarity: 0.664
     Dear Colleagues,  Effective 1 March 2026, the following changes apply to the remote work equipment allowance described in the Remote Work Policy (HR-P...

  2. [HR-POL-2024-003] Remote Work Policy
     Status: current  |  Type: policy  |  Similarity: 0.660
     7. TERMINATION OF REMOTE WORK ARRANGEMENT...

  3. [HR-POL-2024-003] Remote Work Policy
     Status: current  |  Type: policy  |  Similarity: 0.659
     (a) The IT department provides a company laptop, a VPN licence,     and access to the company collaboration tools (Microsoft 365     and Slack).  (b) ...

  4. [HR-FAQ-2025-002] FAQ: Benefits and Allowances
     Status: current  |  Type: faq  |  Similarity: 0.624
     Q1: What health insurance does the company provide? A1: All confirmed employees and up to 3 dependants (

## Building and Applying the Re-ranker
The re-ranker takes ChromaDB's filtered results and reorders them using metadata signals: a boost for current status, a boost by document-type authority (policies and circulars above FAQs), and a boost for recency. We apply it to the remote work query to see if the circular rises above the policy.

In [14]:
TYPE_BOOST = {"policy": 0.08, "sop": 0.05, "circular": 0.06, "faq": 0.00}


def rerank(results):
    """Reorder ChromaDB results by metadata signals."""
    ranked = []
    for chunk_id, doc, meta, dist in zip(
        results["ids"][0], results["documents"][0],
        results["metadatas"][0], results["distances"][0]
    ):
        similarity = 1 - dist
        score = similarity

        if meta["status"] == "current":
            score += 0.10

        score += TYPE_BOOST.get(meta["doc_type"], 0)

        if meta["date_issued"] >= 20250101:
            score += 0.05

        ranked.append({
            "id": chunk_id, "text": doc, "metadata": meta,
            "similarity": similarity, "reranked_score": score
        })

    ranked.sort(key=lambda x: x["reranked_score"], reverse=True)
    return ranked


reranked_trap = rerank(results_trap)

print(f'Query: "{query}" -- AFTER RE-RANKING')
print("=" * 90)
for i, r in enumerate(reranked_trap, 1):
    m = r["metadata"]
    print(f"  {i}. [{m['doc_id']}] {m['title']}")
    print(f"     Status: {m['status']}  |  Type: {m['doc_type']}")
    print(f"     Similarity: {r['similarity']:.3f}  -->  Reranked: {r['reranked_score']:.3f}")
    preview = r["text"].replace('\n', ' ')[:150]
    print(f"     {preview}...")
    print()

Query: "What is the monthly remote work allowance?" -- AFTER RE-RANKING
  1. [HR-CIR-2026-003] Updated Remote Work Equipment Allowance
     Status: current  |  Type: circular
     Similarity: 0.664  -->  Reranked: 0.874
     Dear Colleagues,  Effective 1 March 2026, the following changes apply to the remote work equipment allowance described in the Remote Work Policy (HR-P...

  2. [HR-POL-2024-003] Remote Work Policy
     Status: current  |  Type: policy
     Similarity: 0.660  -->  Reranked: 0.840
     7. TERMINATION OF REMOTE WORK ARRANGEMENT...

  3. [HR-POL-2024-003] Remote Work Policy
     Status: current  |  Type: policy
     Similarity: 0.659  -->  Reranked: 0.839
     (a) The IT department provides a company laptop, a VPN licence,     and access to the company collaboration tools (Microsoft 365     and Slack).  (b) ...

  4. [HR-POL-2024-003] Remote Work Policy
     Status: current  |  Type: policy
     Similarity: 0.597  -->  Reranked: 0.777
     This policy defines the terms

## Comparing All Three Pipelines on a Test Set
We run five questions from the answer key through three configurations -- baseline (no filters), filtered (status=current), and filtered + reranked -- and check whether the expected source appears as the top-1 result.

In [15]:
TEST_QUESTIONS = [
    {"q": "How many days of annual leave do I get?",
     "expected": "HR-POL-2025-006"},
    {"q": "What is the monthly remote work allowance?",
     "expected": "HR-CIR-2026-003"},
    {"q": "Can I carry over unused leave?",
     "expected": "HR-POL-2025-006"},
    {"q": "What is the notice period for a Grade 6 employee?",
     "expected": "HR-SOP-2025-003"},
    {"q": "What changed in the leave policy this year?",
     "expected": "HR-POL-2025-006"},
]

print(f"{'Question':<52} {'Baseline':<18} {'Filtered':<18} {'Reranked':<18}")
print("=" * 106)

scores = {"baseline": 0, "filtered": 0, "reranked": 0}

for tq in TEST_QUESTIONS:
    q, expected = tq["q"], tq["expected"]

    r_base = collection.query(query_texts=[q], n_results=5)
    top1_base = r_base["metadatas"][0][0]["doc_id"]

    r_filt = collection.query(query_texts=[q], n_results=5, where={"status": "current"})
    top1_filt = r_filt["metadatas"][0][0]["doc_id"]

    reranked = rerank(r_filt)
    top1_rank = reranked[0]["metadata"]["doc_id"]

    def mark(doc_id):
        hit = doc_id == expected
        return f"{doc_id} {'OK' if hit else 'MISS'}"

    if top1_base == expected: scores["baseline"] += 1
    if top1_filt == expected: scores["filtered"] += 1
    if top1_rank == expected: scores["reranked"] += 1

    q_short = (q[:50] + "..") if len(q) > 52 else q
    print(f"{q_short:<52} {mark(top1_base):<18} {mark(top1_filt):<18} {mark(top1_rank):<18}")

print("=" * 106)
print(f"{'Score (out of 5)':<52} {scores['baseline']:<18} {scores['filtered']:<18} {scores['reranked']:<18}")

Question                                             Baseline           Filtered           Reranked          
How many days of annual leave do I get?              HR-FAQ-2025-001 MISS HR-FAQ-2025-001 MISS HR-POL-2025-006 OK
What is the monthly remote work allowance?           HR-CIR-2026-003 OK HR-CIR-2026-003 OK HR-CIR-2026-003 OK
Can I carry over unused leave?                       HR-POL-2025-006 OK HR-POL-2025-006 OK HR-POL-2025-006 OK
What is the notice period for a Grade 6 employee?    HR-SOP-2025-003 OK HR-SOP-2025-003 OK HR-SOP-2025-003 OK
What changed in the leave policy this year?          HR-POL-2025-006 OK HR-POL-2025-006 OK HR-POL-2025-006 OK
Score (out of 5)                                     4                  4                  5                 


## The Answer Changes Too
Better retrieval means better answers. We send the same question to the LLM twice -- once with the baseline top-3 chunks (which may include the superseded policy) and once with the filtered-and-reranked top-3 -- and compare the answers.

In [16]:
from openai import OpenAI

llm_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")


def ask_with_chunks(question, chunk_texts):
    context = "\n\n---\n\n".join(chunk_texts)
    response = llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Answer the employee's question using ONLY the provided documents. "
                "If the documents do not contain the answer, say so."
            )},
            {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content


question = "How many days of annual leave do I get?"

# Baseline: unfiltered top-3
r_base = collection.query(query_texts=[question], n_results=3)
print("BASELINE chunks sent to the LLM:")
for m in r_base["metadatas"][0]:
    print(f"  - {m['doc_id']} ({m['status']})")
answer_base = ask_with_chunks(question, r_base["documents"][0])
print(f"\nBASELINE ANSWER:\n{answer_base}\n")

# Full pipeline: filtered + reranked top-3
r_filt = collection.query(query_texts=[question], n_results=5, where={"status": "current"})
reranked = rerank(r_filt)[:3]
print("FULL PIPELINE chunks sent to the LLM:")
for r in reranked:
    print(f"  - {r['metadata']['doc_id']} ({r['metadata']['status']})")
answer_full = ask_with_chunks(question, [r["text"] for r in reranked])
print(f"\nFULL PIPELINE ANSWER:\n{answer_full}")

BASELINE chunks sent to the LLM:
  - HR-FAQ-2025-001 (current)
  - HR-POL-2024-001 (superseded)
  - HR-POL-2025-006 (current)

BASELINE ANSWER:
Confirmed full-time employees get 25 working days per year. This was increased from 20 days in July 2025 (see circular HR-CIR-2025-001).

FULL PIPELINE chunks sent to the LLM:
  - HR-POL-2025-006 (current)
  - HR-POL-2025-006 (current)
  - HR-FAQ-2025-001 (current)

FULL PIPELINE ANSWER:
All confirmed employees are entitled to 25 working days of paid annual leave per calendar year. Employees in their first year of employment receive a pro-rated entitlement calculated from their confirmation date.


## Recap
- **Ingestion:** loaded 15 documents, chunked by section, stored in ChromaDB with six metadata fields
- **The problem:** unfiltered similarity search returned the superseded leave policy (20 days) above the current one (25 days)
- **Status filter:** `where={"status": "current"}` eliminated superseded documents in one line
- **Compound filters:** `$and` combined status, date, and doc_type conditions
- **The trap:** a current policy with outdated amounts -- the circular updated them but the policy was not reissued
- **Re-ranker:** boosted by status, doc_type authority, and recency to fix the ordering
- **Measurement:** five-question test set showed the improvement across all three pipelines

Everything you built here becomes `retrieval.py` in the reference application. Next topic -- **Source Citation in Answers** -- picks up where we left off: once you have the right chunks, how do you cite them in the answer so the employee can verify?